# Chapter 2 - Primer on Probability Modeling

The notebook is a code companion to chapter 2 of the book [Causal AI](https://www.manning.com/books/causal-ai) by [Robert Osazuwa Ness](https://www.linkedin.com/in/osazuwa/).

<a href="https://colab.research.google.com/github/tiagodotai/causalAI/blob/master/book/chapter%202/Chapter_2_Primer_on_Probability_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This code was written with pgmpy version 0.1.24 and pyro version 1.8.6. Install from within a Python environment using:

In [1]:
!pip install pgmpy==0.1.24
!pip install pyro-ppl==1.8.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 732.8/732.8 kB 12.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyro-ppl]


pgmpy depends on pandas. The version of pandas used was 1.5.3. Check your version with:

In [2]:
import pandas as pd
print(pd.__version__)

2.2.3



## Listing 2.1: Implementing a discrete distribution table in pgmpy


In [3]:
import pgmpy
from pgmpy.factors.discrete import DiscreteFactor
dist = DiscreteFactor(
    variables=["X"],    #A
    cardinality=[3],    #B
    values=[.45, .30, .25],    #C
    state_names= {'X': ['1', '2', '3']}    #D
)

#A A list of the names of the variables in the factor
#B The cardinality (number of possible outcomes of each variable in the factor
#C The values each variable in the factor can take
#D Dictionary where the key is the variable name and the value is a list of the names of that variable’s outcomes

print(dist)

+------+----------+
| X    |   phi(X) |
+======+==========+
| X(1) |   0.4500 |
+------+----------+
| X(2) |   0.3000 |
+------+----------+
| X(3) |   0.2500 |
+------+----------+


“phi(X)” is the probability value assigned to each outcome of X.  One thing to note is that these phi values don’t need to sum to one.  For example, I can multiple each probability value by one hundred as follows.

In [4]:
dist = DiscreteFactor(
    variables=["X"],
    cardinality=[3],
    values=[45, 30, 25],
    state_names= {'X': ['1', '2', '3']}
)

print(dist)

+------+----------+
| X    |   phi(X) |
+======+==========+
| X(1) |  45.0000 |
+------+----------+
| X(2) |  30.0000 |
+------+----------+
| X(3) |  25.0000 |
+------+----------+


pgmpy relaxes the constraint to sum to one is because the relaxation is useful in some algorithms.  We can always normalize to obtain proper probability values.

In [5]:
# Normalize takes phi values for each outcome and divides them by their sum to get a probability.
dist.normalize()

print(dist)

+------+----------+
| X    |   phi(X) |
+======+==========+
| X(1) |   0.4500 |
+------+----------+
| X(2) |   0.3000 |
+------+----------+
| X(3) |   0.2500 |
+------+----------+


## Listing 2.2 Modeling a joint distribution in pgmpy

In [6]:
joint = DiscreteFactor(
    variables=['X', 'Y'],   #A
    cardinality=[3, 2],    #B
    values=[.25, .20, .20, .10, .15, .10],   #C
    state_names= {
        'X': ['1', '2', '3'],    #C
        'Y': ['0', '1']    #C
    }
)

#A Now we have two variables instead of one.
#B X has 3 outcomes, Y has 2.
#C You can look at the printed output to see how the values are ordered of values.
#D Now there are two variables, so we name the outcomes for both variables.

print(joint)

+------+------+------------+
| X    | Y    |   phi(X,Y) |
+======+======+============+
| X(1) | Y(0) |     0.2500 |
+------+------+------------+
| X(1) | Y(1) |     0.2000 |
+------+------+------------+
| X(2) | Y(0) |     0.2000 |
+------+------+------------+
| X(2) | Y(1) |     0.1000 |
+------+------+------------+
| X(3) | Y(0) |     0.1500 |
+------+------+------------+
| X(3) | Y(1) |     0.1000 |
+------+------+------------+


The marginalize method will accomplish summing over the specified variables for us.


In [7]:
print(joint.marginalize(variables=['Y'], inplace=False))
print(joint.marginalize(variables=['X'], inplace=False))

+------+----------+
| X    |   phi(X) |
+======+==========+
| X(1) |   0.4500 |
+------+----------+
| X(2) |   0.3000 |
+------+----------+
| X(3) |   0.2500 |
+------+----------+
+------+----------+
| Y    |   phi(Y) |
+======+==========+
| Y(0) |   0.6000 |
+------+----------+
| Y(1) |   0.4000 |
+------+----------+


We can use a division operator on two factors to calculate conditional probabilities. Here, P(X, Y)/P(X) = P(Y|X).

In [8]:
print(joint / dist)

+------+------+------------+
| X    | Y    |   phi(X,Y) |
+======+======+============+
| X(1) | Y(0) |     0.5556 |
+------+------+------------+
| X(1) | Y(1) |     0.4444 |
+------+------+------------+
| X(2) | Y(0) |     0.6667 |
+------+------+------------+
| X(2) | Y(1) |     0.3333 |
+------+------+------------+
| X(3) | Y(0) |     0.6000 |
+------+------+------------+
| X(3) | Y(1) |     0.4000 |
+------+------+------------+


In [9]:
from pgmpy.factors.discrete.CPD import TabularCPD
PYgivenX = TabularCPD(
    variable='Y',    #A
    variable_card=2,    #B
    values=[
        [.25/.45, .20/.30, .15/.25],    #C
        [.20/.45, .10/.30, .10/.25],    #C
    ],
    evidence=['X'],
    evidence_card=[3],
    state_names = {
        'X': ['1', '2', '3'],
        'Y': ['0', '1']
    })

#A Conditional distribution has one variable instead of DiscreteFactor’s list of variables.
#B variable_card is the cardinality of Y
#C Elements of the list correspond to outcomes for Y. Elements of each list correspond to elements of X.

print(PYgivenX)

+------+--------------------+---------------------+------+
| X    | X(1)               | X(2)                | X(3) |
+------+--------------------+---------------------+------+
| Y(0) | 0.5555555555555556 | 0.6666666666666667  | 0.6  |
+------+--------------------+---------------------+------+
| Y(1) | 0.4444444444444445 | 0.33333333333333337 | 0.4  |
+------+--------------------+---------------------+------+


## Listing 2.3 Canonical parameters in Pyro

However, the more direct way in pgmpy to specify a conditional probability distribution table is with the TabularCPD class:

In [10]:
# @title basic pyro just because
import torch
from pyro.distributions import Bernoulli, Categorical, Gamma, Normal

print("Normal")
n = Normal(loc=0.1, scale=4)
print(n.mean)
print(n.stddev)
print(n)

print("\nBernoulli")
b = Bernoulli(probs=0.2)
print(b.probs)
print(b)
print(torch.exp(b.log_prob(torch.tensor(1.0))))
print(b.variance)

print("\nCategorical")
c = Categorical(probs=torch.tensor([.45, .30, .25]))
print(c.probs)


Normal
tensor(0.1000)
tensor(4.)
Normal(loc: 0.10000000149011612, scale: 4.0)

Bernoulli
tensor(0.2000)
Bernoulli(probs: 0.20000000298023224)
tensor(0.2000)
tensor(0.1600)

Categorical
tensor([0.4500, 0.3000, 0.2500])


### Canonical parameters in Pyro

In [11]:
import torch
from pyro.distributions import Bernoulli, Categorical, Gamma, Normal

# The categorical distribution takes a list of probability values, each value corresponding to an outcome.
print(Categorical(probs=torch.tensor([.45, .30, .25])))
print(Normal(loc=0.0, scale=1.0))
print(Bernoulli(probs=0.4))
print(Gamma(concentration=1.0, rate=2.0))

bern = Bernoulli(0.4)
lprob = bern.log_prob(torch.tensor(1.0))

import math
print(math.exp(lprob))


Categorical(probs: torch.Size([3]))
Normal(loc: 0.0, scale: 1.0)
Bernoulli(probs: 0.4000000059604645)
Gamma(concentration: 1.0, rate: 2.0)
0.40000001257540685


## Listing 2.4: Simulating from DiscreteFactor in pgmpy and Pyro

In [34]:
import numpy as np
from pgmpy.factors.discrete import DiscreteFactor
v = [.45, .30, .25, .23]
v = np.array(v) / np.sum(v)
print(v)
dist = DiscreteFactor(
    variables=["X"],
    cardinality=[len(v)],
    values=v,
    state_names= {'X': ['1', '2', '3', 'Tiago']}
)

#n is the number of instances you wish to generate
dist.sample(n=3)


[0.36585366 0.24390244 0.20325203 0.18699187]


,X
0,2
1,1
2,2


In [149]:
joint = DiscreteFactor(
    variables=['X', 'Y'],
    cardinality=[3, 2],
    values=[.25, .20, .20, .10, .15, .10],
    state_names= {
        'X': ['1', '2', '3'],
        'Y': ['0', '1']
    }
)
print(joint)
joint.sample(n=3)



+------+------+------------+
| X    | Y    |   phi(X,Y) |
+======+======+============+
| X(1) | Y(0) |     0.2500 |
+------+------+------------+
| X(1) | Y(1) |     0.2000 |
+------+------+------------+
| X(2) | Y(0) |     0.2000 |
+------+------+------------+
| X(2) | Y(1) |     0.1000 |
+------+------+------------+
| X(3) | Y(0) |     0.1500 |
+------+------+------------+
| X(3) | Y(1) |     0.1000 |
+------+------+------------+


,X,Y
0,1,0
1,2,0
2,1,1


Canonical distributions in pyro also use a method with sample method.

In [145]:
import torch
from pyro.distributions import Categorical
v = [.45, .30, .25, .23]
# Generate 1 sample
one_sample = Categorical(probs=torch.tensor((v+v))).sample()
print("one_sample", one_sample)
# Generate 10 samples
samples = Categorical(probs=torch.tensor([.45, .30, .25])).sample(sample_shape=torch.Size([10]))
print(samples)

one_sample tensor(5)
tensor([0, 2, 0, 2, 0, 0, 1, 0, 0, 0])


## Listing 2.5: Creating a random process in pgmpy and pyro.

In [150]:
from pgmpy.factors.discrete.CPD import TabularCPD
from pgmpy.models import BayesianNetwork
from pgmpy.sampling import BayesianModelSampling

# Values is the probability value assigned to each outcome. The
PZ = TabularCPD(
    variable='Z',
    variable_card=2,
    values=[[.65], [.35]],
    state_names = {
        'Z': ['0', '1']
    })

# The probability values map to the outcomes specified in the state_names argument.
PXgivenZ = TabularCPD(
    variable='X',
    variable_card=2,
    values=[
        [.8, .6],
        [.2, .4],
    ],
    evidence=['Z'],
    evidence_card=[2],
    state_names = {
        'X': ['0', '1'],
        'Z': ['0', '1']
    })

PYgivenX = TabularCPD(
    variable='Y',
    variable_card=3,
    values=[
        [.1, .8],
        [.2, .1],
        [.7, .1],
    ],
    evidence=['X'],
    evidence_card=[2],
    state_names = {
        'Y': ['1', '2', '3'],
        'X': ['0', '1']
    })

model = BayesianNetwork([('Z', 'X'), ('X', 'Y')])
model.add_cpds(PZ, PXgivenZ, PYgivenX)

generator = BayesianModelSampling(model)
generator.forward_sample(size=3)


  0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/pgmpy/sampling/base.py:582: FutureWarning: Passing a DataFrame to DataFrame.from_records is deprecated. Use set_index and/or drop to modify the DataFrame instead.
  df = pd.DataFrame.from_records(samples)


,Z,X,Y
0,1,0,3
1,0,0,3
2,0,0,3


In [155]:
# @title my tabular CPDs

from pgmpy.factors.discrete.CPD import TabularCPD
from pgmpy.models import BayesianNetwork
from pgmpy.sampling import BayesianModelSampling

vpz = [.65, .35]
vpxgivenz = [
    [.8, .6],
    [.2, .4],
]

PZ = TabularCPD(
    variable='Z',
    variable_card=2,
    values=[[.65], [.35]],
    state_names = {
        'Z': ['0', '1']
    })

print(PZ)

PXgivenZ = TabularCPD(
    variable='X',
    variable_card=2,
    values=[
        [.8, .6],
        [.2, .4],
    ],
    evidence=['Z'],
    evidence_card=[2],
    state_names = {
        'X': ['0', '1'],
        'Z': ['0', '1']
    })

print(PXgivenZ)

PYgivenX = TabularCPD(
    variable='Y',
    variable_card=3,
    values=[
        [.1, .8],
        [.2, .1],
        [.7, .1],
    ],
    evidence=['X'],
    evidence_card=[2],
    state_names = {
        'Y': ['1', '2', '3'],
        'X': ['0', '1']
    })

print(PYgivenX)


model = BayesianNetwork([('Z', 'X'), ('X', 'Y')])
model.add_cpds(PZ, PXgivenZ, PYgivenX)

generator = BayesianModelSampling(model)
generator.forward_sample(size=3)

+------+------+
| Z(0) | 0.65 |
+------+------+
| Z(1) | 0.35 |
+------+------+
+------+------+------+
| Z    | Z(0) | Z(1) |
+------+------+------+
| X(0) | 0.8  | 0.6  |
+------+------+------+
| X(1) | 0.2  | 0.4  |
+------+------+------+
+------+------+------+
| X    | X(0) | X(1) |
+------+------+------+
| Y(1) | 0.1  | 0.8  |
+------+------+------+
| Y(2) | 0.2  | 0.1  |
+------+------+------+
| Y(3) | 0.7  | 0.1  |
+------+------+------+


  0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/pgmpy/sampling/base.py:582: FutureWarning: Passing a DataFrame to DataFrame.from_records is deprecated. Use set_index and/or drop to modify the DataFrame instead.
  df = pd.DataFrame.from_records(samples)


,Z,X,Y
0,0,0,3
1,1,0,3
2,1,1,1


In [173]:
generator.forward_sample(size=2)

  0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/pgmpy/sampling/base.py:582: FutureWarning: Passing a DataFrame to DataFrame.from_records is deprecated. Use set_index and/or drop to modify the DataFrame instead.
  df = pd.DataFrame.from_records(samples)


,Z,X,Y
0,1,1,1
1,1,1,1


In [185]:
PZGivenUV = TabularCPD(
    variable='Z',
    variable_card=2,
    evidence=['U', 'V'],
    evidence_card=[2, 2],
    state_names = {
        'Z': ['0', '1'],
        'U': ['0', '1'],
        'V': ['0', '1']
    },
    values=[
        [.1, .9, 20.2, 1.2213],
        [.9, .1, .81, .8],
    ]
)


print(PZGivenUV.normalize(False))



+------+------+------+----------------------+--------------------+
| U    | U(0) | U(0) | U(1)                 | U(1)               |
+------+------+------+----------------------+--------------------+
| V    | V(0) | V(1) | V(0)                 | V(1)               |
+------+------+------+----------------------+--------------------+
| Z(0) | 0.1  | 0.9  | 0.9614469300333175   | 0.6042151090882105 |
+------+------+------+----------------------+--------------------+
| Z(1) | 0.9  | 0.1  | 0.038553069966682535 | 0.3957848909117894 |
+------+------+------+----------------------+--------------------+


## Listing 2.6 Working with combinations of canonical distributions in Pyro

In [186]:
import torch
from pyro.distributions import Bernoulli, Poisson, Gamma

# Represent P(Z) with a Gamma distribution, and sample z.
z = Gamma(7.5, 1.0).sample()    #A
# Represent P(X|Z=z) with a Poisson distribution with location parameter z, and sample x.
x = Poisson(z).sample()    #B
# Represent P(Y|X=x) with a Bernoulli distribution. The probability parameter is a function of y.
y = Bernoulli(x / (5+x)).sample()    #C

print(z, x, y)


tensor(6.4604) tensor(7.) tensor(0.)


## Listing 2.7: Random processes with nuanced control flow in Pyro

In [205]:
import torch
from pyro.distributions import Bernoulli, Poisson, Gamma
z = Gamma(7.5, 1.0).sample()
x = Poisson(z).sample()
# y is calculated as a sum of random coin flips.
# y is generated from P(Y|X=x) because the number of flips depends on x.
y = [torch.tensor(0.0)]
for i in range(int(x)):
    y += [Bernoulli(.5).sample()]
print(z, x, y)

tensor(6.0986) tensor(5.) [tensor(0.), tensor(0.), tensor(1.), tensor(1.), tensor(0.), tensor(1.)]


## Listing 2.8: Using functions for random processes and pyro.sample

In the code below, we generate multiple samples and then use the `mean` method to calculate an average of those samples.

In [ ]:
import torch
import pyro
def random_process():
    z = pyro.sample("z", Gamma(7.5, 1.0))
    x = pyro.sample("x", Poisson(z))
    y = torch.tensor(0.0)
    for i in range(int(x)):
        y += pyro.sample(f"y{i}", Bernoulli(.5))    #A
    return y

generated_samples = generator.forward_sample(size=100)
generated_samples['Y'].apply(int).mean()


  0%|          | 0/3 [00:00<?, ?it/s]

2.21

In [ ]:
generated_samples_ = torch.stack([random_process() for _ in range(100)])
generated_samples_.mean()

tensor(3.7600)

In [ ]:
torch.square(generated_samples_).mean()

tensor(21.4800)

## Listing 2.9: Monte Carlo estimation in Pyro

In [ ]:
import torch
import pyro
from pyro.distributions import Bernoulli, Gamma, Poisson
# This new version of random_process now returns both z and y.
def random_process():
    z = pyro.sample("z", Gamma(7.5, 1.0))
    x = pyro.sample("x", Poisson(z))
    y = torch.tensor(0.0)
    for i in range(int(x)):
        y += pyro.sample(f"{i}", Bernoulli(.5))
    return z, y

# Generate 1000 instances of z and y using a list comprehension.
generated_samples = [random_process() for _ in range(1000)]
# Turn the individual z tensors into a single tensor,
# then calculate the Monte Carlo estimate via the mean method.
z_mean = torch.stack([z for z, _ in generated_samples]).mean()

print(z_mean)

z_given_y = torch.stack([z for z, y in generated_samples if y == 3])

print(z_given_y.mean())


tensor(7.5325)
tensor(6.7901)


In [ ]:
generator.forward_sample(size=10)

  0%|          | 0/3 [00:00<?, ?it/s]

,Z,X,Y
0,0,1,1
1,1,1,1
2,0,0,3
3,0,1,3
4,0,0,1
5,1,0,3
6,0,0,2
7,0,1,3
8,0,0,1
9,1,0,3


## Listing 2.10: Generating IID samples in Pyro

In [ ]:
import pyro
from pyro.distributions import Bernoulli, Poisson, Gamma

def model():
    z = pyro.sample("z", Gamma(7.5, 1.0))
    x = pyro.sample("x", Poisson(z))
    # pyro.plate is a context manager for generating conditional independent
    # samples. This instance of pyro.plate will generate 10 IID samples.
    with pyro.plate('IID', 10):
        # Calling pyro.sample to generates a single outcome y, where y is a
        # tensor of 10 IID samples.
        y = pyro.sample("y", Bernoulli(x / (5+x)))
    return y

model()

tensor([0., 0., 0., 0., 1., 1., 1., 0., 0., 1.])

## Listing 2.11: An example of a data generating process in code form

In [ ]:
def true_dgp(jenny_inclination, brian_inclination, window_strength):    #A
    jenny_throws_rock = jenny_inclination > 0.5    #B
    brian_throws_rock = brian_inclination > 0.5    #B
    if jenny_throws_rock and brian_throws_rock:    #C
        strength_of_impact = 0.8    #C
    elif jenny_throws_rock or brian_throws_rock:    #D
        strength_of_impact = 0.6    #D
    else:    #E
        strength_of_impact = 0.0    #E
    window_breaks = window_strength < strength_of_impact    #F
    return jenny_throws_rock, brian_throws_rock, window_breaks

#A Input variables reflect Jenny and Brian’s inclination to throw and the window strength.
#B Jenny and Brian throw the rock if so inclined.
#C If both Jenny and Brian throw the rock, the total strength of the impact is .8.
#D If either Jenny or Brian throws the rock, the total strength of the impact is .6.
#E Otherwise, no one throws and the strength of impact is 0.
#F If the strength of impact is greater than the strength of the window, the window breaks.

initials = [
    (0.6, 0.31, 0.83),
    (0.48, 0.53, 0.33),
    (0.66, 0.63, 0.75),
    (0.65, 0.66, 0.8),
    (0.48, 0.16, 0.27)
]

data_points = []
# Now we pass each tuple in the initials list as an input to the true_dgp function, generate a data point, and add this to the data_points list.
for jenny_inclination, brian_inclination, window_strength in initials:
    data_points.append(
        true_dgp(
            jenny_inclination, brian_inclination, window_strength
        )
    )

print(data_points)

[(True, False, False), (False, True, True), (True, True, True), (True, True, False), (False, False, False)]
